## Example Spark SQL

### Name: Pablo Camarillo

In [1]:
from pcamarillor.spark_utils import SparkUtils

su = SparkUtils("Ejemplo Spark SQL", "spark://spark-master:7077")
su._spark.sparkContext

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/19 00:19:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


<SparkContext master=spark://spark-master:7077 appName=Ejemplo Spark SQL>

In [2]:
data = [
    (1, "Alice", 29),
    (2, "Bob", 35),
    (3, "Charlie", 41)
]

df = su._spark.createDataFrame(data)
df.printSchema()

root
 |-- _1: long (nullable = true)
 |-- _2: string (nullable = true)
 |-- _3: long (nullable = true)



In [3]:
from pyspark.sql.types import StructField, StructType, IntegerType, StringType

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("nombre", StringType(), True),
    StructField("edad", IntegerType(), True)
])
df = su._spark.createDataFrame(data, schema)
df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- nombre: string (nullable = true)
 |-- edad: integer (nullable = true)



# Smart Factory example

In [14]:
from datetime import datetime
from pyspark.sql.types import TimestampType, FloatType
factory_data = [
    ("M001", datetime(2026, 1, 26, 8, 0, 0), 75.3),
    ("M002", datetime(2026, 1, 26, 8, 5, 0), 68.7),
    ("M001", datetime(2026, 1, 26, 8, 10, 0), 76.1),
    ("M003", datetime(2026, 1, 26, 8, 15, 0), 72.4),
    ("M002", datetime(2026, 1, 26, 8, 20, 0), 69.8),
    ("M001", datetime(2026, 1, 26, 8, 25, 0), 77.5),
    ("M003", datetime(2026, 1, 26, 8, 30, 0), 73.2),
    ("M002", datetime(2026, 1, 26, 8, 35, 0), 70.1),
    ("M001", datetime(2026, 1, 26, 8, 40, 0), 78.0),
    ("M003", datetime(2026, 1, 26, 8, 45, 0), 74.6),
]

factory_schema = StructType([
    StructField("machine_id", StringType(), True),
    StructField("datetime", TimestampType(), True),
    StructField("temp", FloatType(), True)
])

factory_df = su._spark.createDataFrame(factory_data, factory_schema)
factory_df.printSchema()

root
 |-- machine_id: string (nullable = true)
 |-- datetime: timestamp (nullable = true)
 |-- temp: float (nullable = true)



In [13]:
from pyspark.sql.functions import col
df_f = factory_df.filter(col("temp") > 70)
df_f.show()

+----------+-------------------+----+
|machine_id|           datetime|temp|
+----------+-------------------+----+
|      M001|2026-01-26 08:40:00|78.0|
|      M001|2026-01-26 08:25:00|77.5|
|      M001|2026-01-26 08:10:00|76.1|
|      M001|2026-01-26 08:00:00|75.3|
|      M003|2026-01-26 08:45:00|74.6|
+----------+-------------------+----+



In [8]:
print(f"número de elementos en el DataFrame original: {factory_df.count()}")
print(f"número de elementos en el DataFrame filtered: {df_f.count()}")

número de elementos en el DataFrame original: 10
número de elementos en el DataFrame filtered: 8


In [9]:
factory_df.show()

+----------+-------------------+----+
|machine_id|           datetime|temp|
+----------+-------------------+----+
|      M001|2026-01-26 08:00:00|75.3|
|      M002|2026-01-26 08:05:00|68.7|
|      M001|2026-01-26 08:10:00|76.1|
|      M003|2026-01-26 08:15:00|72.4|
|      M002|2026-01-26 08:20:00|69.8|
|      M001|2026-01-26 08:25:00|77.5|
|      M003|2026-01-26 08:30:00|73.2|
|      M002|2026-01-26 08:35:00|70.1|
|      M001|2026-01-26 08:40:00|78.0|
|      M003|2026-01-26 08:45:00|74.6|
+----------+-------------------+----+



In [15]:
factory_df = factory_df.orderBy(col("temp"), ascending=False)
factory_df.show()

+----------+-------------------+----+
|machine_id|           datetime|temp|
+----------+-------------------+----+
|      M001|2026-01-26 08:40:00|78.0|
|      M001|2026-01-26 08:25:00|77.5|
|      M001|2026-01-26 08:10:00|76.1|
|      M001|2026-01-26 08:00:00|75.3|
|      M003|2026-01-26 08:45:00|74.6|
|      M003|2026-01-26 08:30:00|73.2|
|      M003|2026-01-26 08:15:00|72.4|
|      M002|2026-01-26 08:35:00|70.1|
|      M002|2026-01-26 08:20:00|69.8|
|      M002|2026-01-26 08:05:00|68.7|
+----------+-------------------+----+



In [19]:
factory_groupped = factory_df.groupBy(col("machine_id")).count()
factory_groupped.show()

+----------+-----+
|machine_id|count|
+----------+-----+
|      M002|    3|
|      M003|    3|
|      M001|    4|
+----------+-----+



In [21]:
from pyspark.sql.functions import avg, min
agg_df = factory_df.groupBy(col("machine_id")).agg(
    avg("temp").alias("avg_temp"),
    min("temp").alias("min_temp")
)
agg_df.show()



+----------+-----------------+--------+
|machine_id|         avg_temp|min_temp|
+----------+-----------------+--------+
|      M002|69.53333282470703|    68.7|
|      M003|73.39999898274739|    72.4|
|      M001|76.72500038146973|    75.3|
+----------+-----------------+--------+



In [22]:
su._spark.stop()